
# Extended Example:
In a company, the messages to the management team are sent to a common e-mail address of director/deputy, or send to an Skype account which only director has access to it. 
If at least one of the director or deputy receives and checks a message, it will usually be responded.

Facts:
Director is at office with the probability of 50%.
Director is at home with the probability of 20%.
Director is at busyness trip with the probability of 30%.
Deputy is at office with the probability of 70%.

Director usually does not check skype messages, when in office (10% checks).
Director sometimes checks skype messages, when at home (30%). 
Director very unlikely checks skype messages, when at busyness trip (1%).

When one of the director or deputy are in office, the e-mail will be checked with high probability (90%).
When both director and deputy are in office, the e-mail will be checked with a very high probability (95%).
When none of the director or deputy are in office, the e-mail will be checked with a low probability (5%).




Note: all comments from the office BN notebook are valid for this notebook


In [1]:
from pgmpy.factors.discrete import TabularCPD

C:\Users\kuszk\Documents\Visual Studio 2019\Python\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
from pgmpy.models import BayesianNetwork

In [38]:
office_model = BayesianNetwork([('Director', 'Skype'),
                              ('Director', 'Email'),
                              ('Deputy', 'Email'),
                              ('Skype', 'Response'),
                              ('Email', 'Response')])

In [39]:
director_cpd = TabularCPD(
    variable = 'Director',
    variable_card = 3,   # cardinality
    values = [[.5], [.2], [.3]])   # ['office', 'home' ,'bussyness trip']

In [40]:
deputy_cpd = TabularCPD(
    variable = 'Deputy',
    variable_card = 2,   # cardinality
    values = [[.7], [.3]])   # ['office', 'no office']

In [41]:
skype_cpd = TabularCPD(
    variable = 'Skype',
    variable_card = 2,
    values = [[.1, .3, .01],
              [.9, .7, .99]],
    evidence = ['Director'],
    evidence_card = [3])

In [42]:
email_cpd = TabularCPD(
    variable = 'Email',
    variable_card = 2,
    values = [[.95, .4, .9, .1, .9, .05],
              [.05, .6, .1, .9, .1, .95]],
    evidence = ['Director', 'Deputy'],
    evidence_card = [3,2])

In [43]:
response_cpd = TabularCPD(
    variable = 'Response',
    variable_card = 2,
    values = [[.99, .90, .90, 0.0],
              [.01, .10, .10, 1.0]],
    evidence = ['Skype', 'Email'],
    evidence_card = [2,2])

In [44]:
office_model.add_cpds(director_cpd, deputy_cpd, skype_cpd, email_cpd, response_cpd)

In [45]:
office_model.get_cpds()

[<TabularCPD representing P(Director:3) at 0x1b2f915a540>,
 <TabularCPD representing P(Deputy:2) at 0x1b2f8c78ec0>,
 <TabularCPD representing P(Skype:2 | Director:3) at 0x1b2f915a630>,
 <TabularCPD representing P(Email:2 | Director:3, Deputy:2) at 0x1b2f8eb1a60>,
 <TabularCPD representing P(Response:2 | Skype:2, Email:2) at 0x1b2f888c5f0>]

In [46]:
office_model.active_trail_nodes('Director')

{'Director': {'Director', 'Email', 'Response', 'Skype'}}

In [47]:
office_model.local_independencies('Director')

(Director ⟂ Deputy)

In [48]:
office_model.get_independencies()

(Director ⟂ Deputy)
(Director ⟂ Deputy | Skype)
(Director ⟂ Response | Email, Skype)
(Director ⟂ Response | Email, Deputy, Skype)
(Email ⟂ Skype | Director)
(Email ⟂ Skype | Director, Deputy)
(Deputy ⟂ Director, Skype)
(Deputy ⟂ Skype | Director)
(Deputy ⟂ Director | Skype)
(Deputy ⟂ Skype, Response | Director, Email)
(Deputy ⟂ Response | Email, Skype)
(Deputy ⟂ Response | Director, Email, Skype)
(Deputy ⟂ Skype | Director, Email, Response)
(Skype ⟂ Deputy)
(Skype ⟂ Email, Deputy | Director)
(Skype ⟂ Deputy | Director, Email)
(Skype ⟂ Email | Director, Deputy)
(Skype ⟂ Deputy | Director, Email, Response)
(Response ⟂ Deputy | Director, Email)
(Response ⟂ Director, Deputy | Email, Skype)
(Response ⟂ Deputy | Director, Email, Skype)
(Response ⟂ Director | Email, Deputy, Skype)

In [49]:
from pgmpy.inference import VariableElimination

In [50]:
office_infer = VariableElimination(office_model)

In [51]:
prob_response = office_infer.query(variables = ['Response'], joint=False)

In [52]:
print(prob_response['Response'])

+-------------+-----------------+
| Response    |   phi(Response) |
+=============+=================+
| Response(0) |          0.6825 |
+-------------+-----------------+
| Response(1) |          0.3175 |
+-------------+-----------------+


In [53]:
prob_skype = office_infer.query(variables = ['Skype'], joint=False)

In [54]:
print(prob_skype['Skype'])

+----------+--------------+
| Skype    |   phi(Skype) |
+==========+==============+
| Skype(0) |       0.1130 |
+----------+--------------+
| Skype(1) |       0.8870 |
+----------+--------------+


In [55]:
prob_email = office_infer.query(variables = ['Email'], joint=False)

In [56]:
print(prob_email['Email'])

+----------+--------------+
| Email    |   phi(Email) |
+==========+==============+
| Email(0) |       0.7180 |
+----------+--------------+
| Email(1) |       0.2820 |
+----------+--------------+


In [57]:
prob_inoffice = office_infer.query(variables = ['Deputy'], joint=False)

In [58]:
print(prob_inoffice['Deputy'])

+-----------+---------------+
| Deputy    |   phi(Deputy) |
+===========+===============+
| Deputy(0) |        0.7000 |
+-----------+---------------+
| Deputy(1) |        0.3000 |
+-----------+---------------+


In [59]:
prob_inoffice = office_infer.query(variables = ['Director'], joint=False)

In [60]:
print(prob_inoffice['Director'])

+-------------+-----------------+
| Director    |   phi(Director) |
+=============+=================+
| Director(0) |          0.5000 |
+-------------+-----------------+
| Director(1) |          0.2000 |
+-------------+-----------------+
| Director(2) |          0.3000 |
+-------------+-----------------+


In [61]:
prob_skype_response = office_infer.query(
        variables = ['Skype'], joint=False,
        evidence = {'Response':0})

In [62]:
print(prob_skype_response['Skype'])

+----------+--------------+
| Skype    |   phi(Skype) |
+==========+==============+
| Skype(0) |       0.1597 |
+----------+--------------+
| Skype(1) |       0.8403 |
+----------+--------------+


In [63]:
prob_email_response = office_infer.query(
        variables = ['Email'], joint=False,
        evidence = {'Response':0})

In [64]:
print(prob_email_response['Email'])

+----------+--------------+
| Email    |   phi(Email) |
+==========+==============+
| Email(0) |       0.9575 |
+----------+--------------+
| Email(1) |       0.0425 |
+----------+--------------+


In [65]:
prob_deputy_response = office_infer.query(
        variables = ['Deputy'], joint=False,
        evidence = {'Response':0})

In [66]:
print(prob_deputy_response['Deputy'])

+-----------+---------------+
| Deputy    |   phi(Deputy) |
+===========+===============+
| Deputy(0) |        0.8716 |
+-----------+---------------+
| Deputy(1) |        0.1284 |
+-----------+---------------+


In [67]:
prob_director_response = office_infer.query(
        variables = ['Director'], joint=False,
        evidence = {'Response':0})

In [68]:
print(prob_director_response['Director'])

+-------------+-----------------+
| Director    |   phi(Director) |
+=============+=================+
| Director(0) |          0.5370 |
+-------------+-----------------+
| Director(1) |          0.2062 |
+-------------+-----------------+
| Director(2) |          0.2568 |
+-------------+-----------------+


In [69]:
prob_director_skype_noemail = office_infer.query(
        variables = ['Director'], joint=False,
        evidence = {'Skype':0, 'Email': 1})

In [70]:
print(prob_director_skype_noemail['Director'])

+-------------+-----------------+
| Director    |   phi(Director) |
+=============+=================+
| Director(0) |          0.3337 |
+-------------+-----------------+
| Director(1) |          0.6332 |
+-------------+-----------------+
| Director(2) |          0.0331 |
+-------------+-----------------+


In [71]:
prob_deputy_skype_noemail = office_infer.query(
        variables = ['Deputy'], joint=False,
        evidence = {'Skype':0, 'Email': 1})

In [72]:
print(prob_deputy_skype_noemail['Deputy'])

+-----------+---------------+
| Deputy    |   phi(Deputy) |
+===========+===============+
| Deputy(0) |        0.1912 |
+-----------+---------------+
| Deputy(1) |        0.8088 |
+-----------+---------------+


In [73]:
prob_response_skype_noemail = office_infer.query(
        variables = ['Response'], joint=False,
        evidence = {'Skype':0, 'Email': 1})

In [74]:
print(prob_response_skype_noemail['Response'])

+-------------+-----------------+
| Response    |   phi(Response) |
+=============+=================+
| Response(0) |          0.9000 |
+-------------+-----------------+
| Response(1) |          0.1000 |
+-------------+-----------------+


In [75]:
prob_director_response_skype = office_infer.query(
        variables = ['Director'], joint=False,
        evidence = {'Response':0 ,'Skype':0 })

In [76]:
print(prob_director_response_skype['Director'])

+-------------+-----------------+
| Director    |   phi(Director) |
+=============+=================+
| Director(0) |          0.4454 |
+-------------+-----------------+
| Director(1) |          0.5283 |
+-------------+-----------------+
| Director(2) |          0.0264 |
+-------------+-----------------+


In [77]:
prob_deputy_response_skype = office_infer.query(
        variables = ['Deputy'], joint=False,
        evidence = {'Response':0 ,'Skype':0 })

In [78]:
print(prob_deputy_response_skype['Deputy'])

+-----------+---------------+
| Deputy    |   phi(Deputy) |
+===========+===============+
| Deputy(0) |        0.7135 |
+-----------+---------------+
| Deputy(1) |        0.2865 |
+-----------+---------------+


In [79]:
prob_email_response_skype = office_infer.query(
        variables = ['Email'], joint=False,
        evidence = {'Response':0 ,'Skype':0 })

In [45]:
print(prob_email_response_skype['Email'])

+----------+--------------+
| Email    |   phi(Email) |
+==========+==============+
| Email(0) |       0.7339 |
+----------+--------------+
| Email(1) |       0.2661 |
+----------+--------------+


In [46]:
prob_director_response_skype_noemail = office_infer.query(
        variables = ['Director'],joint=False,
        evidence = {'Response':0,'Skype':0, 'Email':1})

Eliminating: Deputy: 100%|██████████| 1/1 [00:00<00:00, 330.26it/s]


In [47]:
print(prob_director_response_skype_noemail['Director'])

+-------------+-----------------+
| Director    |   phi(Director) |
+=============+=================+
| Director(0) |          0.3337 |
+-------------+-----------------+
| Director(1) |          0.6332 |
+-------------+-----------------+
| Director(2) |          0.0331 |
+-------------+-----------------+


In [48]:
prob_deputy_response_skype_noemail = office_infer.query(
        variables = ['Deputy'],joint=False,
        evidence = {'Response':0,'Skype':0, 'Email':1})

Eliminating: Director: 100%|██████████| 1/1 [00:00<00:00, 332.22it/s]


In [49]:
print(prob_deputy_response_skype_noemail['Deputy'])

+-----------+---------------+
| Deputy    |   phi(Deputy) |
+===========+===============+
| Deputy(0) |        0.1912 |
+-----------+---------------+
| Deputy(1) |        0.8088 |
+-----------+---------------+
